# Check the Colab training environment

Inspect the Python version, pinned package versions, imports, dependency consistency and CUDA availability in the active runtime. This standalone check leaves the environment unchanged. The final cell reports READY only when the checks pass and lists any differences from the project requirements.

In [ ]:
import sys
import platform
import importlib
import importlib.metadata as metadata
import subprocess
import json

# Snapshot of the == pins in train_historydata/requirements.txt.
# Keep these synchronized if that requirements.txt changes.
REQUIRED = {'torch': '2.5.1', 'numpy': '1.26.4', 'pandas': '2.2.3', 'scipy': '1.13.1', 'networkx': '3.4.2', 'scikit-learn': '1.5.2', 'tqdm': '4.67.1', 'matplotlib': '3.9.2', 'seaborn': '0.13.2', 'plotly': '5.24.1', 'ipywidgets': '8.1.5'}
IMPORT_NAMES = {
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "networkx": "networkx",
    "scikit-learn": "sklearn",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
}

python_ok = (3, 10) <= sys.version_info[:2] <= (3, 12)
print(f"Python: {platform.python_version()} — {'OK' if python_ok else 'MISMATCH (project supports Python 3.10–3.12)'}")

version_rows = []
version_ok = True
for package, expected in REQUIRED.items():
    try:
        actual = metadata.version(package)
    except metadata.PackageNotFoundError:
        actual = None
    ok = actual == expected
    version_ok &= ok
    version_rows.append({"package": package, "required": expected,
                         "installed": actual or "MISSING", "status": "OK" if ok else "MISMATCH"})
print("\nPinned package versions:")
for row in version_rows:
    print(f"{row['status']:8s} {row['package']:16s} required={row['required']:10s} installed={row['installed']}")

import_ok = True
import_rows = []
for package, module_name in IMPORT_NAMES.items():
    try:
        module = importlib.import_module(module_name)
        detail = getattr(module, "__version__", "imported")
        import_rows.append({"package": package, "status": "OK", "detail": str(detail)})
    except Exception as exc:
        import_ok = False
        import_rows.append({"package": package, "status": "IMPORT FAILED",
                            "detail": f"{type(exc).__name__}: {exc}"})
print("\nImport checks:")
for row in import_rows:
    print(f"{row['status']:14s} {row['package']:16s} {row['detail']}")

try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    gpu_name = torch.cuda.get_device_name(0) if cuda_ok else None
    cuda_version = torch.version.cuda
    print(f"\nPyTorch: {torch.__version__}; built CUDA: {cuda_version}")
    print(f"CUDA GPU: {gpu_name if cuda_ok else 'NOT AVAILABLE'}")
except Exception as exc:
    cuda_ok, gpu_name, cuda_version = False, None, None
    print(f"\nPyTorch/GPU check failed: {type(exc).__name__}: {exc}")

# Report dependency conflicts without installing or changing anything.
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"],
                           capture_output=True, text=True, check=False)
pip_check_text = (pip_check.stdout + pip_check.stderr).strip() or "No dependency conflicts reported."
print("\npip check:")
print(pip_check_text)

ready = python_ok and version_ok and import_ok and cuda_ok and pip_check.returncode == 0
report = {
    "status": "READY" if ready else "NOT READY",
    "python": platform.python_version(),
    "python_supported": python_ok,
    "pinned_versions_match": version_ok,
    "imports_pass": import_ok,
    "cuda_available": cuda_ok,
    "gpu_name": gpu_name,
    "cuda_build": cuda_version,
    "pip_check_returncode": pip_check.returncode,
    "versions": version_rows,
    "imports": import_rows,
    "pip_check": pip_check_text,
    "packages_installed_or_changed": False,
    "training_started": False,
}
print("\nSUMMARY")
print(json.dumps(report, indent=2))
if not ready:
    print("\nEnvironment check did not pass. No packages were installed or changed. Review mismatches before training.")
